In [ ]:
import pandas as pd 
import torch
import numpy as np
import random
import os
from matplotlib import pyplot as plt

def set_seed(seed: int = 42) -> None:
    """Sets the random seed for reproducibility across PyTorch, NumPy, and Python's random module."""
    os.environ['PYTHONHASHSEED'] = str(seed)  # For Python's hash seed
    torch.manual_seed(seed)  # For PyTorch's CPU and CUDA RNGs
    torch.cuda.manual_seed(seed)  # For CUDA devices specifically
    torch.cuda.manual_seed_all(seed) # For all CUDA devices if multiple are used
    np.random.seed(seed)  # For NumPy's random number generator
    random.seed(seed)  # For Python's built-in random module

    # For deterministic algorithms in PyTorch (optional, but recommended for full reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Example usage:
set_seed(123)

In [ ]:
import sys
import os

# Add project root (parent of "src") to Python path
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_OhioT1DM_patient_split

Ohio_PATH = "..\data\\raw\\" 
data = load_OhioT1DM_patient_split(path=Ohio_PATH, look_back=128)

In [ ]:
X_train, X_val, X_test, y_reg_train, y_reg_val, y_reg_test, y_clf_train, y_clf_val, y_clf_test = data

In [ ]:
x = X_test

In [ ]:
import glob 
import re

generated_PATH = "..\data\\generated\\synth_**.pt"

generated_paths  = glob.glob(generated_PATH) 

pattern = r"synth_(\w+)\.pt"  # capture word characters between 'synth_' and '.pt'

generated_data = {}

for path in generated_paths: 
    
    data = torch.load(path, weights_only=False)
    file_name = path.split("\\")[-1]
    match = re.search(pattern, file_name)
    generator = match.group(1)
    data = data.clip(40, 400)
    generated_data[generator] = np.array(data.detach().cpu()).reshape(-1, 128)
        
    print(f"synth data generated by {generator} loaded: {file_name} .")



In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
from scipy.spatial.distance import cdist 
from sklearn.manifold import trustworthiness

def frechet_distance(X, Y):
    mu_x, mu_y = X.mean(0), Y.mean(0)
    cov_x, cov_y = np.cov(X, rowvar=False), np.cov(Y, rowvar=False)
    covmean = sqrtm(cov_x.dot(cov_y)).real
    return np.sum((mu_x - mu_y)**2) + np.trace(cov_x + cov_y - 2*covmean)

# Step 1: PCA (retain most variance)
pca = PCA(n_components=0.95)
pca.fit(X_train)
real_pca = pca.transform(x)
umap_model = umap.UMAP(n_neighbors=30, n_components=12, learning_rate=0.1, min_dist=0.1, metric='cosine', random_state=42, n_epochs=1500, init="random")
umap_model.fit(X_train)
real_umap = umap_model.transform(real_pca) 
df = pd.DataFrame(real_umap)
mask = df.isna().values
mask = np.any(mask, axis = 1)
real_umap = df.dropna().values

# Step 3: Verify stability
trust_score = trustworthiness(real_pca[~mask], real_umap, n_neighbors=12)
print(f"Embedding trustworthiness: {trust_score:.3f}")
if trust_score < 0.8:
    print("⚠️ Warning: UMAP may be distorting structure. Consider tuning parameters.")

In [ ]:
def coverage_ratio_func(X_real, X_synth, threshold=0.1):
    """
    Calculate what fraction of real samples are covered by synthetic samples.
    
    Args:
        X_real: Real data points (N x D)
        X_synth: Synthetic data points (M x D)
        threshold: Distance threshold for "coverage"
    
    Returns:
        Fraction of real samples with at least one synthetic neighbor within threshold
    """
    coverage = np.mean([
        np.min(np.linalg.norm(X_synth - r[None, :], axis=-1)) < threshold
        for r in X_real
    ])
    return coverage

def precision_recall_kynkaanniemi(X_real, X_synth, k=3):
    """
    Standard Precision-Recall for distributions.
    
    Most cited implementation: Kynkäänniemi et al., NeurIPS 2019
    "Improved Precision and Recall Metric for Assessing Generative Models"
    
    Uses k-th NN distance as LOCAL threshold (adapts to density)
    """
    from sklearn.neighbors import NearestNeighbors
    
    # Build k-NN models
    # k+1 because first neighbor is the point itself
    nbrs_real = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(X_real)
    nbrs_synth = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(X_synth)
    
    # === STEP 1: Define "manifolds" using k-th NN distance ===
    # For each real point, find its k-th nearest neighbor
    real_distances, _ = nbrs_real.kneighbors(X_real)
    real_radii = real_distances[:, k]  # k-th column = k-th NN distance
    
    # For each synth point, find its k-th nearest neighbor
    synth_distances, _ = nbrs_synth.kneighbors(X_synth)
    synth_radii = synth_distances[:, k]
    
    # === STEP 2: Precision ===
    # For each synthetic sample, find nearest real sample
    synth_to_real_dist, synth_to_real_idx = nbrs_real.kneighbors(X_synth, n_neighbors=1)
    synth_to_real_dist = synth_to_real_dist.flatten()
    synth_to_real_idx = synth_to_real_idx.flatten()
    
    # A synth sample has good "precision" if it's within the real manifold
    # i.e., distance to nearest real < that real point's k-th NN radius
    precision = np.mean(synth_to_real_dist < real_radii[synth_to_real_idx])
    
    # === STEP 3: Recall (Coverage) ===
    # For each real sample, find nearest synthetic sample
    real_to_synth_dist, real_to_synth_idx = nbrs_synth.kneighbors(X_real, n_neighbors=1)
    real_to_synth_dist = real_to_synth_dist.flatten()
    real_to_synth_idx = real_to_synth_idx.flatten()
    
    # A real sample is "covered" if it's within the synthetic manifold
    # i.e., distance to nearest synth < that synth point's k-th NN radius
    recall = np.mean(real_to_synth_dist < synth_radii[real_to_synth_idx])
    
    return precision, recall


for model, data in generated_data.items(): 
    # Step 1: PCA (retain most variance)
    synth_pca = pca.transform(data[:len(x)])
    # Step 2: UMAP (for nonlinear manifold)
    synth_umap = umap_model.transform(synth_pca) 
    df_synth = pd.DataFrame(synth_umap)
    synth_umap = df_synth.dropna().values
    
    fid_score = frechet_distance(real_pca, synth_pca)
    
    precision, recall = precision_recall_kynkaanniemi(real_umap, synth_umap, k=3)
    
    #coverage_ratio = coverage_ratio_func(real_umap, synth_umap, threshold=0.1)
    print(f"FID-like diversity score {model}: {fid_score:.3f}")
    print(f"Prec-Recl in Umap Space {model}: {precision:.3f}, {recall:.3f}")
    
